# 📊 EduGap India — Government School Infrastructure & Teacher Load Analysis

### Identifying critical infrastructure gaps and teacher pressure across Indian states using UDISE+ data
---

In [79]:
import pandas as pd

file_22 = "/workspaces/EduGap-India/Dataset/Table 2.2 State wise highlights of the UDISE+ 2023-24 data Schools, Enrolments and Teachers - page 31.csv"
file_25_1 = "/workspaces/EduGap-India/Dataset/Table 2.5 State wise highlights of the UDISE+ 2023-24 data Infrastructure - page 34.csv"
file_25_2 = "/workspaces/EduGap-India/Dataset/Table 2.5 (continued) State wise highlights of the UDISE+ 2023-24 data Infrastructure - page 35.csv"

df_22 = pd.read_csv(file_22)
df_25_1 = pd.read_csv(file_25_1)
df_25_2 = pd.read_csv(file_25_2)

---
## 🎯 Problem Statement

Millions of students in Indian government schools study without access to basic infrastructure such as electricity, functional toilets, and digital resources.

While this data exists in the UDISE+ database, it is rarely transformed into actionable insights.

This project aims to:
- Identify infrastructure gaps across states
- Measure teacher workload using PTR (Pupil-Teacher Ratio)
- Detect high-risk states where poor infrastructure and high teaching load coexist
---


In [80]:
df_22.columns = df_22.columns.str.strip().str.replace("\n", " ").str.replace("  ", " ")
df_25_1.columns = df_25_1.columns.str.strip().str.replace("\n", " ").str.replace("  ", " ")
df_25_2.columns = df_25_2.columns.str.strip().str.replace("\n", " ").str.replace("  ", " ")

In [81]:
df_25 = pd.concat([df_25_1, df_25_2], ignore_index=True)

## 📂 Dataset Description

Source: UDISE+ (Unified District Information System for Education)

Datasets used:
- Table 2.2 → Schools, Enrolments, Teachers
- Table 2.5 → Infrastructure availability

Key variables:
- Total Schools
- Total Enrolments
- Electricity availability
- Functional toilets
- Teacher counts

---

In [82]:
print(df_22.head())
print(df_25.head())

print(df_22.columns)
print(df_25.columns)

              India/State/UT Total number of Schools  \
0                        NaN                     NaN   
1                        (1)                     (2)   
2                      India                 1471891   
3  Andaman & Nicobar Islands                     412   
4             Andhra Pradesh                   61373   

  Pupil Enrolments Teacher Ratio Average Teachers Per School Teachers  \
0                            NaN                                  NaN   
1                            (3)                                  (4)   
2                      248045828                              9807600   
3                          72119                                 5750   
4                        8741885                               338293   

  Average Enrolments Per School Schools with Zero Enrolments  \
0                           NaN                          NaN   
1                           (5)                          (6)   
2                            25 

## 🧹 Data Cleaning

The raw dataset required significant preprocessing due to:

- Multi-row headers from PDF conversion
- Misaligned columns
- Duplicate rows
- Missing values

Steps performed:
- Removed invalid header rows
- Fixed column naming issues
- Merged datasets on state level
- Removed duplicate and null entries
- Converted numerical columns to correct data types

---

In [83]:
df_25 = pd.concat([df_25_1, df_25_2], ignore_index=True)

df = pd.merge(df_22, df_25, on="India/State/UT")

In [84]:
print(df.shape)
print(df.head())

(78, 34)
  India/State/UT Total number of Schools Pupil Enrolments Teacher Ratio  \
0            NaN                     NaN                            NaN   
1            NaN                     NaN                            NaN   
2            (1)                     (2)                            (3)   
3            (1)                     (2)                            (3)   
4          India                 1471891                      248045828   

  Average Teachers Per School Teachers Average Enrolments Per School  \
0                                  NaN                           NaN   
1                                  NaN                           NaN   
2                                  (4)                           (5)   
3                                  (4)                           (5)   
4                              9807600                            25   

  Schools with Zero Enrolments Teachers in Schools having zero Enrolments  \
0                          NaN

Removing the garbage 

In [85]:
df = df[df["India/State/UT"].notna()]
df = df[~df["India/State/UT"].isin(["(1)", "India/State/UT"])]

In [86]:
print(df.shape)
print(df.head())

(74, 34)
              India/State/UT Total number of Schools  \
4                      India                 1471891   
5                      India                 1471891   
6  Andaman & Nicobar Islands                     412   
7  Andaman & Nicobar Islands                     412   
8             Andhra Pradesh                   61373   

  Pupil Enrolments Teacher Ratio Average Teachers Per School Teachers  \
4                      248045828                              9807600   
5                      248045828                              9807600   
6                          72119                                 5750   
7                          72119                                 5750   
8                        8741885                               338293   

  Average Enrolments Per School Schools with Zero Enrolments  \
4                            25                            7   
5                            25                            7   
6                      

In [87]:
df = df.drop_duplicates(subset="India/State/UT")

In [88]:
print(df.shape)
print(df.head())

(37, 34)
               India/State/UT Total number of Schools  \
4                       India                 1471891   
6   Andaman & Nicobar Islands                     412   
8              Andhra Pradesh                   61373   
10          Arunachal Pradesh                    3490   
12                      Assam                   56630   

   Pupil Enrolments Teacher Ratio Average Teachers Per School Teachers  \
4                       248045828                              9807600   
6                           72119                                 5750   
8                         8741885                               338293   
10                         323717                                24700   
12                        6922533                               342199   

   Average Enrolments Per School Schools with Zero Enrolments  \
4                             25                            7   
6                             13                           14   
8       

In [89]:
df = df[df["India/State/UT"] != "India"]

In [90]:
print(df.shape)
print(df.head())

(36, 34)
               India/State/UT Total number of Schools  \
6   Andaman & Nicobar Islands                     412   
8              Andhra Pradesh                   61373   
10          Arunachal Pradesh                    3490   
12                      Assam                   56630   
14                      Bihar                   94686   

   Pupil Enrolments Teacher Ratio Average Teachers Per School Teachers  \
6                           72119                                 5750   
8                         8741885                               338293   
10                         323717                                24700   
12                        6922533                               342199   
14                       21348149                               657063   

   Average Enrolments Per School Schools with Zero Enrolments  \
6                             13                           14   
8                             26                            6   
10      

In [91]:
df = df.apply(lambda col: pd.to_numeric(col, errors='coerce'))
print(df.dtypes)

India/State/UT                                                  float64
Total number of Schools                                           int64
Pupil Enrolments Teacher Ratio                                    int64
Average Teachers Per School Teachers                              int64
Average Enrolments Per School                                     int64
Schools with Zero Enrolments                                      int64
Teachers in Schools having zero Enrolments                        int64
Schools with Single Teachers                                      int64
Enrolments in Single Teacher Schools                              int64
Unnamed: 9                                                        int64
Unnamed: 10                                                       int64
Total Schools                                                     int64
Number of School having                                           int64
Library/ Book Bank/ Reading Corner                              

In [92]:
df["India/State/UT"] = df["India/State/UT"].astype(str)

In [93]:
print(df["India/State/UT"].head())
print(df.dtypes["India/State/UT"])

6     NaN
8     NaN
10    NaN
12    NaN
14    NaN
Name: India/State/UT, dtype: str
str


In [94]:
df["India/State/UT"] = df_22["India/State/UT"]
print(df["India/State/UT"].head())
print(df.dtypes["India/State/UT"])

6                                    Assam
8                               Chandigarh
10    Dadra & Nagar Haveli and Daman & Diu
12                                     Goa
14                                 Haryana
Name: India/State/UT, dtype: str
str


In [95]:
df = df.drop(columns=["Unnamed: 9", "Unnamed: 10"])

In [96]:
print(df.shape)
print(df.columns)

(36, 32)
Index(['India/State/UT', 'Total number of Schools',
       'Pupil Enrolments Teacher Ratio',
       'Average Teachers Per School Teachers', 'Average Enrolments Per School',
       'Schools with Zero Enrolments',
       'Teachers in Schools having zero Enrolments',
       'Schools with Single Teachers', 'Enrolments in Single Teacher Schools',
       'Total Schools', 'Number of School having',
       'Library/ Book Bank/ Reading Corner', 'Playground', 'Digital Library',
       'Kitchen Garden', 'Girls' Toilet', 'Functional Girls' Toilet',
       'Boys' Toilet', 'Functional Boys' Toilet', 'Electricity',
       'Functional Electricity', 'Solar Panel', 'Computer facility',
       'Functional Computer facility for Pedagogical purposes',
       'Internet Facility', 'Drinking Water', 'Functional Drinking Water',
       'Hand wash facility', 'Functional Rainwater Harvesting System',
       'Conducting Medical Checkup of Students in Last Academic Year', 'Ramp',
       'Ramp and Handrail

## ⚙️ Feature Engineering

To extract meaningful insights, new metrics were created:

### 🔌 Electricity Ratio
Proportion of schools with electricity

In [97]:
df["electricity_ratio"] = df["Functional Electricity"] / df["Total Schools"]
print(df[["India/State/UT", "electricity_ratio"]].head())

                          India/State/UT  electricity_ratio
6                                  Assam           0.033981
8                             Chandigarh           0.076956
10  Dadra & Nagar Haveli and Daman & Diu           0.117479
12                                   Goa           0.065195
14                               Haryana           0.092104


Validating the metric

In [98]:
print(df[["India/State/UT", "Functional Electricity", "Total Schools", "electricity_ratio"]].head(10))

                          India/State/UT  Functional Electricity  \
6                                  Assam                      14   
8                             Chandigarh                    4723   
10  Dadra & Nagar Haveli and Daman & Diu                     410   
12                                   Goa                    3692   
14                               Haryana                    8721   
16                       Jammu & Kashmir                     173   
18                             Karnataka                    3428   
20                                Ladakh                      62   
22                        Madhya Pradesh                    1835   
24                               Manipur                      51   

    Total Schools  electricity_ratio  
6             412           0.033981  
8           61373           0.076956  
10           3490           0.117479  
12          56630           0.065195  
14          94686           0.092104  
16            230

In [99]:
print(df[["India/State/UT", "Electricity", "Functional Electricity"]].head(10))

                          India/State/UT  Electricity  Functional Electricity
6                                  Assam          379                      14
8                             Chandigarh        61213                    4723
10  Dadra & Nagar Haveli and Daman & Diu         2023                     410
12                                   Goa        48850                    3692
14                               Haryana        74183                    8721
16                       Jammu & Kashmir          230                     173
18                             Karnataka        51347                    3428
20                                Ladakh          432                      62
22                        Madhya Pradesh         5497                    1835
24                               Manipur         1487                      51


In [100]:
df["electricity_ratio"] = df["Electricity"] / df["Total number of Schools"]

In [101]:
print(df[["India/State/UT", "Electricity", "Total number of Schools", "electricity_ratio"]].head())

                          India/State/UT  Electricity  \
6                                  Assam          379   
8                             Chandigarh        61213   
10  Dadra & Nagar Haveli and Daman & Diu         2023   
12                                   Goa        48850   
14                               Haryana        74183   

    Total number of Schools  electricity_ratio  
6                       412           0.919903  
8                     61373           0.997393  
10                     3490           0.579656  
12                    56630           0.862617  
14                    94686           0.783463  


In [102]:
df_sorted = df.sort_values(by="electricity_ratio", ascending=False)

print(df_sorted[["India/State/UT", "electricity_ratio"]].head(10))

     India/State/UT  electricity_ratio
22   Madhya Pradesh           1.000000
24          Manipur           1.000000
20           Ladakh           1.000000
16  Jammu & Kashmir           1.000000
42              NaN           1.000000
58              NaN           1.000000
26          Mizoram           0.999124
60              NaN           0.998723
38      West Bengal           0.998235
8        Chandigarh           0.997393


In [103]:
df = df[df["India/State/UT"].notna()]
df_sorted = df.sort_values(by="electricity_ratio", ascending=False)
print(df_sorted[["India/State/UT", "electricity_ratio"]].head(10))

     India/State/UT  electricity_ratio
20           Ladakh           1.000000
16  Jammu & Kashmir           1.000000
22   Madhya Pradesh           1.000000
24          Manipur           1.000000
26          Mizoram           0.999124
38      West Bengal           0.998235
8        Chandigarh           0.997393
28           Odisha           0.993792
30           Punjab           0.992258
36    Uttar Pradesh           0.982654


In [104]:
df.to_csv("cleaned_education_data.csv", index=False)

In [105]:
df_sorted_low = df.sort_values(by="electricity_ratio", ascending=True)
print(df_sorted_low[["India/State/UT", "electricity_ratio"]].head(10))

                          India/State/UT  electricity_ratio
10  Dadra & Nagar Haveli and Daman & Diu           0.579656
14                               Haryana           0.783463
32                                Sikkim           0.859771
12                                   Goa           0.862617
18                             Karnataka           0.906950
6                                  Assam           0.919903
34                             Telangana           0.920112
36                         Uttar Pradesh           0.982654
30                                Punjab           0.992258
28                                Odisha           0.993792


### 🚽 Girls Toilet Availability

In [106]:
df["girls_toilet_ratio"] = df["Functional Girls' Toilet"] / df["Total number of Schools"]
print(df[["India/State/UT", "girls_toilet_ratio"]].head())

                          India/State/UT  girls_toilet_ratio
6                                  Assam            0.997573
8                             Chandigarh            0.872713
10  Dadra & Nagar Haveli and Daman & Diu            0.906590
12                                   Goa            0.907540
14                               Haryana            0.924815


### 🏗️ Infrastructure Score
Average of key infrastructure indicators

In [107]:
df["infra_score"] = (df["electricity_ratio"] + df["girls_toilet_ratio"]) / 2
print(df[["India/State/UT", "infra_score"]].head())

                          India/State/UT  infra_score
6                                  Assam     0.958738
8                             Chandigarh     0.935053
10  Dadra & Nagar Haveli and Daman & Diu     0.743123
12                                   Goa     0.885079
14                               Haryana     0.854139


In [108]:
worst_states = df.sort_values(by="infra_score", ascending=True)

print(worst_states[["India/State/UT", "infra_score"]].head(10)) 

                          India/State/UT  infra_score
10  Dadra & Nagar Haveli and Daman & Diu     0.743123
14                               Haryana     0.854139
32                                Sikkim     0.874136
12                                   Goa     0.885079
18                             Karnataka     0.922803
22                        Madhya Pradesh     0.928688
8                             Chandigarh     0.935053
34                             Telangana     0.944238
6                                  Assam     0.958738
36                         Uttar Pradesh     0.968037


In [109]:
print(worst_states[["India/State/UT", "infra_score", "Pupil Enrolments Teacher Ratio"]].head(10))

                          India/State/UT  infra_score  \
10  Dadra & Nagar Haveli and Daman & Diu     0.743123   
14                               Haryana     0.854139   
32                                Sikkim     0.874136   
12                                   Goa     0.885079   
18                             Karnataka     0.922803   
22                        Madhya Pradesh     0.928688   
8                             Chandigarh     0.935053   
34                             Telangana     0.944238   
6                                  Assam     0.958738   
36                         Uttar Pradesh     0.968037   

    Pupil Enrolments Teacher Ratio  
10                          323717  
14                        21348149  
32                         2629949  
12                         6922533  
18                         5776548  
22                         4506578  
8                          8741885  
34                         7143255  
6                            72119  
36

In [110]:
print(df[["India/State/UT", "Pupil Enrolments Teacher Ratio"]].head(10))

                          India/State/UT  Pupil Enrolments Teacher Ratio
6                                  Assam                           72119
8                             Chandigarh                         8741885
10  Dadra & Nagar Haveli and Daman & Diu                          323717
12                                   Goa                         6922533
14                               Haryana                        21348149
16                       Jammu & Kashmir                          265706
18                             Karnataka                         5776548
20                                Ladakh                          141282
22                        Madhya Pradesh                         4506578
24                               Manipur                          304735


In [111]:
df.rename(columns={
    "Pupil Enrolments Teacher Ratio": "Total Enrolments"
}, inplace=True)

In [112]:
print(df[["India/State/UT", "Total Enrolments"]].head())

                          India/State/UT  Total Enrolments
6                                  Assam             72119
8                             Chandigarh           8741885
10  Dadra & Nagar Haveli and Daman & Diu            323717
12                                   Goa           6922533
14                               Haryana          21348149


### 👨‍🏫 PTR (Pupil-Teacher Ratio)

In [113]:
df["PTR"] = df["Total Enrolments"] / df["Average Teachers Per School Teachers"]
print(df[["India/State/UT", "PTR"]].head())

                          India/State/UT        PTR
6                                  Assam  12.542435
8                             Chandigarh  25.841164
10  Dadra & Nagar Haveli and Daman & Diu  13.105951
12                                   Goa  20.229554
14                               Haryana  32.490262


In [114]:
print(worst_states[["India/State/UT", "infra_score"]]
      .merge(df[["India/State/UT", "PTR"]], on="India/State/UT")
      .head(10))

                         India/State/UT  infra_score        PTR
0  Dadra & Nagar Haveli and Daman & Diu     0.743123  13.105951
1                               Haryana     0.854139  32.490262
2                                Sikkim     0.874136  15.743861
3                                   Goa     0.885079  20.229554
4                             Karnataka     0.922803  20.719474
5                        Madhya Pradesh     0.928688  28.082042
6                            Chandigarh     0.935053  25.841164
7                             Telangana     0.944238  34.576797
8                                 Assam     0.958738  12.542435
9                         Uttar Pradesh     0.968037  27.483634


## 🧠 Key Insights

- States like **Haryana and Telangana** show high teacher pressure combined with weaker infrastructure
- Some states (e.g., Dadra & Nagar Haveli) show low infrastructure but manageable teacher load
- High PTR regions indicate potential overburdened teaching systems
- Infrastructure alone does not determine educational stress — combined analysis is necessary

---

In [115]:
df["high_risk"] = (df["infra_score"] < 0.9) & (df["PTR"] > 25)
print(df[["India/State/UT", "infra_score", "PTR", "high_risk"]])

                          India/State/UT  infra_score        PTR  high_risk
6                                  Assam     0.958738  12.542435      False
8                             Chandigarh     0.935053  25.841164      False
10  Dadra & Nagar Haveli and Daman & Diu     0.743123  13.105951      False
12                                   Goa     0.885079  20.229554      False
14                               Haryana     0.854139  32.490262       True
16                       Jammu & Kashmir     0.993478  25.955456      False
18                             Karnataka     0.922803  20.719474      False
20                                Ladakh     0.995370  28.284685      False
22                        Madhya Pradesh     0.928688  28.082042      False
24                               Manipur     0.996301  20.880841      False
26                               Mizoram     0.977949  29.175540      False
28                                Odisha     0.973594  22.317820      False
30          

In [117]:
df["high_risk"] = (df["infra_score"] < 0.95) & (df["PTR"] > 25)
print(df[["India/State/UT", "infra_score", "PTR", "high_risk"]])

                          India/State/UT  infra_score        PTR  high_risk
6                                  Assam     0.958738  12.542435      False
8                             Chandigarh     0.935053  25.841164       True
10  Dadra & Nagar Haveli and Daman & Diu     0.743123  13.105951      False
12                                   Goa     0.885079  20.229554      False
14                               Haryana     0.854139  32.490262       True
16                       Jammu & Kashmir     0.993478  25.955456      False
18                             Karnataka     0.922803  20.719474      False
20                                Ladakh     0.995370  28.284685      False
22                        Madhya Pradesh     0.928688  28.082042       True
24                               Manipur     0.996301  20.880841      False
26                               Mizoram     0.977949  29.175540      False
28                                Odisha     0.973594  22.317820      False
30          

In [118]:
df.to_csv("education_analysis_ready.csv", index=False)